# Audit and robustness checks (run before submitting)
Everything is real data. **Run All** (about 5-10 minutes); the last cell zips the results to `Desktop/Q2/outputs_audit.zip` - send me that zip.

| part | what it does |
|---|---|
| A | data integrity: counts, duplicates, ID uniqueness, date sanity, profit sanity, **leakage screen** (any feature that is suspiciously predictive on its own) |
| B | **independent recomputation** of headline numbers with a differently written conformal implementation (p-value formulation), compared with the values printed in the manuscript (PASS/FAIL) |
| C | robustness: calibration-set resampling plus month-block bootstrap for coverage, model-seed variance, dropping the earliest vintages, winsorised/trimmed profit |

Needs `extra_data/lending_club_36m_wide.csv` and `extra_data/prosper/prosperLoanData.csv` (downloaded automatically if missing); `extra_data/payments.csv` enables the Lending Club profit checks.

In [1]:
import os, sys, subprocess, importlib, warnings, json, hashlib, time, traceback, contextlib, shutil, urllib.request
from pathlib import Path
def ensure(pkg, mod=None):
    try: importlib.import_module(mod or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
ensure('lightgbm'); ensure('statsmodels'); ensure('scikit-learn', 'sklearn')
import numpy as np, pandas as pd, lightgbm as lgb, matplotlib
matplotlib.use(os.environ.get('MPL_BACKEND', matplotlib.get_backend()))
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings('ignore'); plt.style.use('seaborn-v0_8-whitegrid')
def find_q2():
    if os.environ.get('Q2_DIR'): return Path(os.environ['Q2_DIR'])
    home = Path.home()
    for d in [home/'Desktop', home/'OneDrive'/'Desktop', *sorted(home.glob('OneDrive*/Desktop'))]:
        if (d/'Q2').is_dir(): return d/'Q2'
Q2 = find_q2(); assert Q2 is not None, "No 'Q2' folder on the Desktop; set Q2 = Path(r'...') here."
OUT = Q2/'outputs_audit'; TAB, FIG = OUT/'tables', OUT/'figures'; TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(exist_ok=True)
SEED, ALPHA = 42, 0.10; FAILED = []; T0 = time.time()
@contextlib.contextmanager
def section(name):
    t = time.time(); print(f"\n{'=' * 10} {name} {'=' * 10}")
    try: yield
    except Exception:
        FAILED.append(name); print(f"!!! SECTION FAILED: {name}"); traceback.print_exc()
    finally: print(f"[{name}: {time.time() - t:.0f}s | total {(time.time() - T0) / 60:.1f} min]")
def save(df, name): df.round(5).to_csv(TAB/f'{name}.csv', index=False)
def savefig(name):
    for ext in ('png', 'pdf'): plt.savefig(FIG/f'{name}.{ext}', dpi=180, bbox_inches='tight')
def conf_q(scores, alpha):
    s = np.sort(np.asarray(scores)); n = len(s); k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else s[k - 1]
def true_scores(p, y): return np.where(y == 1, 1 - p, p)
def conformal_sets(p, y, ca, alpha=ALPHA, nbin=5):
    sc = true_scores(p, y); q = conf_q(sc[ca], alpha); q0, q1 = conf_q(sc[ca & (y == 0)], alpha), conf_q(sc[ca & (y == 1)], alpha)
    edges = np.quantile(p[ca], np.linspace(0, 1, nbin + 1)[1:-1]); b = np.digitize(p, edges)
    qb = np.array([conf_q(sc[ca & (b == k)], alpha) for k in range(nbin)])[b]
    return {'static': np.column_stack([p <= q, (1 - p) <= q]), 'class-conditional': np.column_stack([p <= q0, (1 - p) <= q1]), 'bin-conditional': np.column_stack([p <= qb, (1 - p) <= qb])}
def cov_stats(S, y):
    hit = S[np.arange(len(y)), y]; ap, rj = S[:, 0] & ~S[:, 1], S[:, 1] & ~S[:, 0]
    return dict(cov=hit.mean(), cov_default=hit[y == 1].mean() if (y == 1).any() else np.nan, size=S.sum(1).mean(), share_review=1 - ap.mean() - rj.mean(), share_reject_only=rj.mean())
def tiers(S): ap = S[:, 0] & ~S[:, 1]; rj = S[:, 1] & ~S[:, 0]; return ap, ~(ap | rj), rj
def wilson(k, n): return proportion_confint(int(k), int(n), method='wilson')
print("Q2:", Q2)

Q2: /Users/author/Desktop/Q2


In [2]:
# ---------- 0. extraction: Prosper file -> analysis frame ----------
with section('0. data'):
    PF = Q2/'extra_data'/'prosper'/'prosperLoanData.csv'; PF.parent.mkdir(parents=True, exist_ok=True)
    if not PF.exists():
        url = "https://raw.githubusercontent.com/chukwume/prosperloandata/master/prosperLoanData.csv"; print("downloading", url)
        urllib.request.urlretrieve(url, PF)
    sha = hashlib.sha256(PF.read_bytes()).hexdigest(); print("SHA-256:", sha, "| matches expected:", sha == 'fce13408ac22e8201363e71e1a16d9795305f4aaf8c0f67efcd80424bc5b9e02')
    d = pd.read_csv(PF, low_memory=False); print("raw rows:", len(d))
    dt = lambda c: pd.to_datetime(d[c], format='ISO8601', errors='coerce')
    d['orig'], d['closed'], d['listed'], d['first_cl'] = dt('LoanOriginationDate'), dt('ClosedDate'), dt('ListingCreationDate'), dt('FirstRecordedCreditLine')
    keep = d['LoanStatus'].isin(['Completed', 'Chargedoff', 'Defaulted']) & (d['Term'] == 36) & (d['orig'] >= '2006-01-01') & (d['orig'] <= '2011-03-31')
    f = d[keep].sort_values('orig', kind='stable').reset_index(drop=True); print(f"resolved 36-month loans, issued 2006 to Mar-2011: {len(f):,}  default rate {f['LoanStatus'].ne('Completed').mean():.2%}")
    f['default'] = f['LoanStatus'].ne('Completed').astype(int); f['year'], f['month'] = f['orig'].dt.year, f['orig'].dt.month
    f['tm'] = (f['year'] - 2006) * 12 + f['month']; f['reveal'] = np.maximum(((f['closed'].dt.year - 2006) * 12 + f['closed'].dt.month).values, f['tm'].values + 1)
    f['funded'] = f['LoanOriginalAmount'].astype(float)
    f['profit'] = f['LP_InterestandFees'] + f['LP_ServiceFees'] + f['LP_CollectionFees'] + f['LP_NonPrincipalRecoverypayments'] - f['LP_NetPrincipalLoss']
    f['roi'] = f['profit'] / f['funded']
    gmap = {'AA': 0, 'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'HR': 6}
    f['grade_ord'] = f['ProsperRating (Alpha)'].map(gmap).fillna(f['CreditGrade'].map(gmap))
    f['fico'] = (f['CreditScoreRangeLower'] + f['CreditScoreRangeUpper']) / 2; f['dti'] = f['DebtToIncomeRatio'].clip(0, 1.5)
    f['log_income'] = np.log1p(f['StatedMonthlyIncome'].clip(0, 100000)); f['log_amount'] = np.log1p(f['funded']); f['rate'] = f['BorrowerRate']
    f['homeowner'] = f['IsBorrowerHomeowner'].astype(float); f['employment'] = f['EmploymentStatus'].fillna('NA'); f['purpose'] = f['ListingCategory (numeric)'].astype(str); f['state'] = f['BorrowerState'].fillna('NA')
    f['bankcard_util'] = f['BankcardUtilization'].clip(0, 2); f['log_revbal'] = np.log1p(f['RevolvingCreditBalance'].clip(lower=0)); f['log_amt_delinq'] = np.log1p(f['AmountDelinquent'].clip(lower=0)); f['log_avail_bc'] = np.log1p(f['AvailableBankcardCredit'].clip(lower=0))
    f['credit_hist_yrs'] = (f['listed'] - f['first_cl']).dt.days / 365.25
    for a, b in (('open_lines', 'OpenCreditLines'), ('curr_delinq', 'CurrentDelinquencies'), ('delinq_7y', 'DelinquenciesLast7Years'), ('pub_rec_10y', 'PublicRecordsLast10Years'), ('inq_6m', 'InquiriesLast6Months'),
                 ('total_inq', 'TotalInquiries'), ('total_trades', 'TotalTrades'), ('never_delinq_pct', 'TradesNeverDelinquent (percentage)'), ('curr_lines', 'CurrentCreditLines')): f[a] = f[b]
    f['prior_prosper_loans'] = f['TotalProsperLoans'].fillna(0)
    CORE = ['fico', 'dti', 'log_income', 'log_amount', 'homeowner', 'inq_6m', 'bankcard_util', 'curr_delinq', 'open_lines', 'log_revbal']
    BUREAU = ['log_amt_delinq', 'delinq_7y', 'pub_rec_10y', 'total_inq', 'log_avail_bc', 'total_trades', 'never_delinq_pct', 'credit_hist_yrs', 'curr_lines', 'prior_prosper_loans']
    miss = f[CORE + BUREAU + ['rate', 'grade_ord']].isna().groupby(f['year']).mean().T.round(2); print("\nMissing share by year:"); print(miss.to_string())
    drop = [c for c in CORE + BUREAU if f.loc[f.year <= 2008, c].isna().mean() > 0.5]; CORE = [c for c in CORE if c not in drop]; BUREAU = [c for c in BUREAU if c not in drop]; print("dropped (mostly missing in training years):", drop)
    FS = {'P0 with lender rate': (CORE + ['rate'], ['employment', 'purpose']), 'P1 no lender score': (CORE, ['employment', 'purpose']),
          'P3 applicant-only rich': (CORE + BUREAU, ['employment', 'purpose', 'state']), 'P4 everything': (CORE + BUREAU + ['rate', 'grade_ord'], ['employment', 'purpose', 'state'])}
    Xc = {k: pd.get_dummies(f[n + c], columns=c, drop_first=True) for k, (n, c) in FS.items()}
    y, yr, tm, pi, fund, roi = f['default'].values, f['year'].values, f['tm'].values, f['profit'].values, f['funded'].values, f['roi'].values
    T1 = f.groupby('year').agg(N=('default', 'size'), default_rate=('default', 'mean'), mean_rate=('rate', 'mean'), mean_fico=('fico', 'mean'), mean_dti=('dti', 'mean'), mean_amount=('funded', 'mean'),
                               profit_per_loan=('profit', 'mean'), return_on_funded=('profit', lambda s: s.sum() / f.loc[s.index, 'funded'].sum())).reset_index(); save(T1, 'P1_data_summary_by_year'); print(); print(T1.round(3).to_string(index=False))
    g = f.groupby('default').apply(lambda x: x['profit'].sum() / x['funded'].sum()); print(f"\nprofit per $ funded: repaid {g[0]:+.3f}, default {g[1]:+.3f} -> break-even default probability ~ {g[0] / (g[0] - g[1]):.1%}")
    save(pd.DataFrame([dict(profit_per_dollar_repaid=g[0], profit_per_dollar_default=g[1], breakeven_default_prob=g[0] / (g[0] - g[1]))]), 'P2_breakeven')
    lag = ((f['reveal'] - f['tm']))
    desc = pd.DataFrame({'outcome': np.where(y == 1, 'default', 'repaid'), 'months_to_label': lag}).groupby('outcome')['months_to_label'].describe(percentiles=[.1, .25, .5, .75, .9]).reset_index(); save(desc, 'P3_label_lag_by_outcome'); print("\nMonths from origination to label known (exact ClosedDate):"); print(desc.round(1).to_string(index=False))


========== 0. data ==========
SHA-256: fce13408ac22e8201363e71e1a16d9795305f4aaf8c0f67efcd80424bc5b9e02 | matches expected: True


raw rows: 113937
resolved 36-month loans, issued 2006 to Mar-2011: 37,934  default rate 32.15%

Missing share by year:
year                 2006  2007  2008  2009  2010  2011
fico                 0.10  0.00  0.00  0.00  0.00   0.0
dti                  0.01  0.02  0.08  0.10  0.12   0.1
log_income           0.00  0.00  0.00  0.00  0.00   0.0
log_amount           0.00  0.00  0.00  0.00  0.00   0.0
homeowner            0.00  0.00  0.00  0.00  0.00   0.0
inq_6m               0.11  0.00  0.00  0.00  0.00   0.0
bankcard_util        1.00  0.15  0.00  0.00  0.00   0.0
curr_delinq          0.11  0.00  0.00  0.00  0.00   0.0
open_lines           1.00  0.15  0.00  0.00  0.00   0.0
log_revbal           1.00  0.15  0.00  0.00  0.00   0.0
log_amt_delinq       1.00  0.15  0.00  0.00  0.00   0.0
delinq_7y            0.15  0.00  0.00  0.00  0.00   0.0
pub_rec_10y          0.11  0.00  0.00  0.00  0.00   0.0
total_inq            0.19  0.00  0.00  0.00  0.00   0.0
log_avail_bc         1.00  0.14  0.00  0.

In [3]:
# ---------- 1. model zoo and temporal designs ----------
LGB_REG = dict(n_estimators=250, learning_rate=0.03, num_leaves=15, max_depth=4, min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=10.0, random_state=SEED, verbose=-1)
MODELS = ['LGBM default', 'LGBM regularised', 'Logistic (L2)']
def fit_predict(kind, X, mtr):
    if kind == 'LGBM default': m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=SEED, verbose=-1)
    elif kind == 'LGBM regularised': m = lgb.LGBMClassifier(**LGB_REG)
    else:
        Xf = X.astype('float32'); miss_ = [c for c in Xf.columns if Xf.loc[mtr, c].isna().any()]; ind = Xf[miss_].isna().astype('float32').add_suffix('_na')
        Xf = pd.concat([Xf.fillna(Xf.loc[mtr].median().fillna(0)), ind], axis=1); mu, sd = Xf.loc[mtr].mean().fillna(0), Xf.loc[mtr].std().replace(0, 1).fillna(1); Z = np.nan_to_num(((Xf - mu) / sd).values)
        return LogisticRegression(C=0.1, max_iter=3000).fit(Z[mtr], y[mtr]).predict_proba(Z)[:, 1]
    return m.fit(X[mtr], y[mtr]).predict_proba(X)[:, 1]
def fit_reg(X, target, mtr): return lgb.LGBMRegressor(**LGB_REG).fit(X[mtr], target[mtr]).predict(X)
DESIGNS = {'B': dict(label='B: train 2006-08, calibrate 2009, test 2010-11Q1', train_end=2008, ca=2009, tests=[2010, 2011]),
           'A': dict(label='A: train 2006-07, calibrate 2008, test 2009-11Q1', train_end=2007, ca=2008, tests=[2009, 2010, 2011])}
tlabel = lambda yy: '2011Q1' if yy == 2011 else str(yy)


In [4]:
# ---------- Lending Club frame ----------
AUD = []
def audit(part, item, value, note=''): AUD.append(dict(part=part, item=item, value=value, note=note))
# ---- If the automatic search below cannot find your files, paste their FULL paths here (keep the quotes), e.g.
#      MANUAL_WIDE = '/Users/yourname/Downloads/lending_club_36m_wide.csv'
MANUAL_WIDE = ''
MANUAL_PAYMENTS = ''
HAS_LC = False; HAS_LCPAY = False
def find_file(patterns, extra_roots=()):
    for root in [Q2, *extra_roots, Q2.parent, Path.home()/'Downloads', Path.home()/'Documents']:
        if not Path(root).exists(): continue
        for pat in patterns:
            hits = [h for h in sorted(Path(root).rglob(pat)) if h.is_file()]
            if hits: return hits[0]
    return None
with section('LC load'):
    WIDE = Path(MANUAL_WIDE) if MANUAL_WIDE else find_file(['lending_club_36m_wide*.csv'])
    if WIDE is None: raise FileNotFoundError("lending_club_36m_wide.csv not found under Desktop/Q2, Desktop, Downloads or Documents. Put it in Desktop/Q2/extra_data/ (it is the file written by Extract_Extra_Data.ipynb) and rerun.")
    print("Lending Club wide file:", WIDE)
    W = pd.read_csv(WIDE, dtype={'id': str}); LC = W[W.year <= 2015].reset_index(drop=True)
    PAYF = Path(MANUAL_PAYMENTS) if MANUAL_PAYMENTS else find_file(['payments*.csv']); HAS_LCPAY = False
    if PAYF is not None:
        _p = pd.read_csv(PAYF, dtype={'id': str})
        if {'id', 'total_pymnt'} <= set(_p.columns):
            LC = LC.merge(_p[['id', 'total_pymnt']].drop_duplicates('id'), on='id', how='left'); LC['profit'] = LC['total_pymnt'] - LC['funded_amnt']; HAS_LCPAY = True; print("payments file:", PAYF)
    if not HAS_LCPAY: print("!! payments.csv not found or lacks id/total_pymnt: the Lending Club profit checks will be skipped (everything else still runs)")
    yL, yrL = LC['default'].values, LC['year'].values
    LC_NUM = ['loan_amnt', 'annual_inc', 'dti', 'fico_score', 'revol_util', 'delinq_2yrs', 'emp_length_yrs', 'int_rate']; LC_CAT = ['home_ownership', 'verification_status', 'purpose']
    XL = pd.get_dummies(LC[LC_NUM + LC_CAT], columns=LC_CAT, drop_first=True); XL1 = XL.drop(columns=['int_rate'])
    def fit_lgb(kind, X, mtr, seed=42):
        if kind == 'default': m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=seed, verbose=-1)
        else: m = lgb.LGBMClassifier(**{**LGB_REG, 'random_state': seed, 'min_child_samples': 200})
        return m.fit(X[mtr], yL[mtr]).predict_proba(X)[:, 1]
    HAS_LC = True; print(f"Lending Club frame: {len(LC):,} loans; payments available: {HAS_LCPAY}")




========== LC load ==========
Lending Club wide file: /Users/author/Desktop/Q2/extra_data/lending_club_36m_wide.csv


payments file: /Users/author/Desktop/Q2/extra_data/payments.csv
Lending Club frame: 582,113 loans; payments available: True
[LC load: 1s | total 0.0 min]


In [5]:
# ---------- stop here with a clear message if the Lending Club data were not loaded ----------
if not HAS_LC:
    raise RuntimeError("STOP: the Lending Club file was not loaded (see the message printed under 'LC load' above). Fix: put lending_club_36m_wide.csv in Desktop/Q2/extra_data/, or set MANUAL_WIDE = '/full/path/lending_club_36m_wide.csv' near the top of the previous cell, then run all again. Nothing below can run without it.")
print('Lending Club data loaded - continuing.')


Lending Club data loaded - continuing.


In [6]:
# ---------- A. data integrity ----------
with section('A. data integrity'):
    audit('A-LC', 'loans (36-month, resolved, issued <= 2015)', len(LC), 'manuscript: 582,113'); audit('A-LC', 'default rate', yL.mean(), 'manuscript: 13.48%')
    audit('A-LC', 'duplicate loan ids', int(LC['id'].duplicated().sum())); audit('A-LC', 'rows with a missing core feature', int(LC[LC_NUM].isna().any(axis=1).sum()))
    audit('A-LC', 'fully duplicated feature rows', int(LC[LC_NUM + LC_CAT + ['issue_d']].duplicated().sum()), 'identical loans in the same month: expected to be small')
    audit('A-LC', 'issue years', f"{LC['year'].min()}-{LC['year'].max()}"); audit('A-LC', 'train / calibration / test loans', f"{(yrL <= 2011).sum():,} / {(yrL == 2012).sum():,} / {(yrL >= 2013).sum():,}", 'manuscript: 28,243 / 41,703 / 512,167')
    mj = find_file(['raw_manifest.json'])
    if mj is not None: m_ = json.loads(mj.read_text()); audit('A-LC', 'matches extraction manifest (rows)', len(LC) == m_['outputs'][0]['rows'])
    if HAS_LCPAY:
        pr = LC['profit'].values; fu = LC['funded_amnt'].values
        audit('A-LC', 'payments matched', LC['total_pymnt'].notna().mean()); audit('A-LC', 'share of repaid loans with negative profit', float((pr[yL == 0] < 0).mean()), 'should be small')
        audit('A-LC', 'share of defaulted loans with positive profit', float((pr[yL == 1] > 0).mean()), 'recoveries exceeding principal: should be rare')
        audit('A-LC', 'return on funded, repaid / default', f"{pr[yL == 0].sum() / fu[yL == 0].sum():+.3f} / {pr[yL == 1].sum() / fu[yL == 1].sum():+.3f}", 'manuscript: +0.152 / -0.364')
    # Prosper
    audit('A-PR', 'raw rows', len(d), 'expected 113,937'); audit('A-PR', 'duplicate ListingKey in raw', int(d['ListingKey'].duplicated().sum()))
    audit('A-PR', 'analysis loans (36-month, resolved, 2006 to Mar-2011)', len(f), 'manuscript: 37,934'); audit('A-PR', 'default rate', y.mean(), 'manuscript: 32.2%')
    audit('A-PR', 'loans with ClosedDate before origination', int((f['closed'] < f['orig']).sum()), 'handled: label time is set to at least the month after origination')
    audit('A-PR', 'resolved loans without ClosedDate', int(f['closed'].isna().sum())); audit('A-PR', 'duplicate loan keys in analysis frame', int(f['ListingKey'].duplicated().sum()))
    audit('A-PR', 'repaid loans with negative profit', float((pi[y == 0] < 0).mean())); audit('A-PR', 'defaulted loans with positive profit', float((pi[y == 1] > 0).mean()))
    audit('A-PR', 'return on funded, repaid / default', f"{pi[y == 0].sum() / fund[y == 0].sum():+.3f} / {pi[y == 1].sum() / fund[y == 1].sum():+.3f}", 'manuscript: +0.200 / -0.474')
    print(pd.DataFrame(AUD).to_string(index=False))

# ---------- A3. leakage screen: univariate AUC on training years ----------
LEAK = []
with section('A3. leakage screen'):
    for name, Xd, yy, mtr in (('LC', XL, yL, yrL <= 2011), ('Prosper', Xc['P4 everything'], y, yr <= 2008)):
        for c in Xd.columns:
            v = Xd.loc[mtr, c].astype(float); ok = v.notna().values
            if ok.sum() < 200 or v[ok].nunique() < 2: continue
            a = roc_auc_score(yy[mtr][ok], v[ok]); LEAK.append(dict(portfolio=name, feature=c, univariate_auc=a, strength=abs(a - 0.5)))
    LEAK = pd.DataFrame(LEAK).sort_values('strength', ascending=False); save(LEAK, 'A3_leakage_screen')
    print(LEAK.groupby('portfolio').head(6).round(3).to_string(index=False)); print("\nFlag threshold: |AUC-0.5| > 0.30 (a single origination-time field that predicts default this well deserves a look):", LEAK[LEAK.strength > 0.30][['portfolio', 'feature', 'univariate_auc']].round(3).values.tolist() or 'none')
    audit('A3', 'features flagged (|AUC-0.5| > 0.30)', int((LEAK.strength > 0.30).sum()), 'int_rate / lender grade are the strongest by design: they are known at origination')


========== A. data integrity ==========


part                                                  item                     value                                                               note
A-LC            loans (36-month, resolved, issued <= 2015)                    582113                                                manuscript: 582,113
A-LC                                          default rate                  0.134842                                                 manuscript: 13.48%
A-LC                                    duplicate loan ids                         0                                                                   
A-LC                      rows with a missing core feature                         0                                                                   
A-LC                         fully duplicated feature rows                         0            identical loans in the same month: expected to be small
A-LC                                           issue years                 2007-2015    

portfolio             feature  univariate_auc  strength
  Prosper                rate           0.704     0.204
  Prosper                fico           0.342     0.158
  Prosper           grade_ord           0.653     0.153
  Prosper           total_inq           0.643     0.143
       LC            int_rate           0.638     0.138
  Prosper              inq_6m           0.633     0.133
       LC          fico_score           0.378     0.122
  Prosper        log_avail_bc           0.389     0.111
       LC          revol_util           0.580     0.080
       LC          annual_inc           0.428     0.072
       LC                 dti           0.530     0.030
       LC home_ownership_RENT           0.526     0.026

Flag threshold: |AUC-0.5| > 0.30 (a single origination-time field that predicts default this well deserves a look): none
[A3. leakage screen: 1s | total 0.1 min]


In [7]:
# ---------- B. independent recomputation (p-value formulation) vs manuscript values ----------
CHK = []
def pv_sets(p, cal_true_scores):
    """Split conformal via p-values: label k is in the set iff (1 + #{cal scores >= s(x,k)}) / (n+1) > alpha."""
    s = np.sort(cal_true_scores); n = len(s); out = []
    for k in (0, 1):
        sk = p if k == 0 else 1 - p; ge = n - np.searchsorted(s, sk, side='left'); out.append((1 + ge) / (n + 1) > ALPHA)
    return np.column_stack(out)
def check(item, expected, got, tol=0.0011): CHK.append(dict(item=item, manuscript=expected, recomputed=round(float(got), 4), abs_diff=round(abs(expected - got), 4), result='PASS' if abs(expected - got) <= tol else 'CHECK'))
with section('B. independent recomputation'):
    pL = fit_lgb('default', XL, yrL <= 2011); scL = true_scores(pL, yL); caL = yrL == 2012; SL = pv_sets(pL, scL[caL])
    for yy, exp in ((2013, .9178), (2014, .9025), (2015, .8900)):
        m = yrL == yy; check(f'LC static coverage {yy} (baseline)', exp, SL[m][np.arange(m.sum()), yL[m]].mean())
    m = yrL == 2015; check('LC static coverage of defaulters 2015 (alpha 0.10)', 0.236, SL[m & (yL == 1)][:, 1].mean())
    g = {yy: 100 * (yL[yrL == yy].mean() - pL[yrL == yy].mean()) for yy in (2013, 2014, 2015)}; check('LC actual - predicted default 2015 (points)', 2.65, g[2015], 0.02); check('LC actual - predicted default 2013 (points)', -1.6, g[2013], 0.06)
    # Prosper design B baseline
    X0 = Xc['P0 with lender rate']; pP = fit_predict('LGBM default', X0, yr <= 2008); scP = true_scores(pP, y); caP = yr == 2009; SP = pv_sets(pP, scP[caP])
    for yy, exp in ((2010, .914), (2011, .926)):
        m = yr == yy; check(f'Prosper static coverage {yy} (design B baseline)', exp, SP[m][np.arange(m.sum()), y[m]].mean())
    q0 = conf_q(scP[caP & (y == 0)], ALPHA); q1 = conf_q(scP[caP & (y == 1)], ALPHA)
    for yy, exp in ((2010, .904), (2011, .976)):
        m = yr == yy; check(f'Prosper class-conditional coverage of defaulters {yy}', exp, ((1 - pP[m & (y == 1)]) <= q1).mean())
    check('Prosper actual default 2010', .167, y[yr == 2010].mean(), 0.0011); check('Prosper mean predicted default 2010', .275, pP[yr == 2010].mean(), 0.0011)
    CHK = pd.DataFrame(CHK); save(CHK, 'B_manuscript_checks'); print(CHK.to_string(index=False)); audit('B', 'manuscript numbers recomputed independently: PASS / total', f"{(CHK.result == 'PASS').sum()} / {len(CHK)}")


========== B. independent recomputation ==========


                                                 item  manuscript  recomputed  abs_diff result
                   LC static coverage 2013 (baseline)      0.9178      0.9178    0.0000   PASS
                   LC static coverage 2014 (baseline)      0.9025      0.9025    0.0000   PASS
                   LC static coverage 2015 (baseline)      0.8900      0.8900    0.0000   PASS
   LC static coverage of defaulters 2015 (alpha 0.10)      0.2360      0.2360    0.0000   PASS
          LC actual - predicted default 2015 (points)      2.6500      2.6480    0.0020   PASS
          LC actual - predicted default 2013 (points)     -1.6000     -1.6082    0.0082   PASS
     Prosper static coverage 2010 (design B baseline)      0.9140      0.9137    0.0003   PASS
     Prosper static coverage 2011 (design B baseline)      0.9260      0.9258    0.0002   PASS
Prosper class-conditional coverage of defaulters 2010      0.9040      0.9037    0.0003   PASS
Prosper class-conditional coverage of defaulters 2

In [8]:
# ---------- C. robustness ----------
ROB = []
def cov_interval(sc_cal, sc_test, months_test, B=2000, seed=SEED):
    """Coverage of the true-label score <= q, resampling the calibration set (with replacement) and test issue-month blocks."""
    rng = np.random.default_rng(seed); ms = np.unique(months_test); per = [np.sort(sc_test[months_test == m]) for m in ms]; out = np.empty(B)
    for b in range(B):
        q = conf_q(rng.choice(sc_cal, len(sc_cal), replace=True), ALPHA); pick = rng.integers(0, len(ms), len(ms))
        hit = sum(np.searchsorted(per[i], q, side='right') for i in pick); tot = sum(len(per[i]) for i in pick); out[b] = hit / tot
    return np.percentile(out, [2.5, 97.5]), np.mean(out < 1 - ALPHA)
with section('C1. calibration + month-block resampling'):
    mon = LC['month'].values
    for yy in (2013, 2014, 2015):
        m = yrL == yy; (lo, hi), pbelow = cov_interval(scL[caL], scL[m], mon[m]); ROB.append(dict(check='coverage interval (calibration + month blocks)', portfolio='Lending Club', period=str(yy), estimate=SL[m][np.arange(m.sum()), yL[m]].mean(), lo=lo, hi=hi, extra=f'share of draws below 0.90: {pbelow:.2f}'))
    mP = f['month'].values
    for yy in (2010, 2011):
        m = yr == yy; (lo, hi), pbelow = cov_interval(scP[caP], scP[m], mP[m]); ROB.append(dict(check='coverage interval (calibration + month blocks)', portfolio='Prosper B', period=tlabel(yy), estimate=SP[m][np.arange(m.sum()), y[m]].mean(), lo=lo, hi=hi, extra=f'share of draws below 0.90: {pbelow:.2f}'))
    pA = fit_predict('LGBM default', X0, yr <= 2007); scA = true_scores(pA, y); caA = yr == 2008; SA = pv_sets(pA, scA[caA])
    for yy in (2009, 2010, 2011):
        m = yr == yy; (lo, hi), pbelow = cov_interval(scA[caA], scA[m], mP[m]); ROB.append(dict(check='coverage interval (calibration + month blocks)', portfolio='Prosper A', period=tlabel(yy), estimate=SA[m][np.arange(m.sum()), y[m]].mean(), lo=lo, hi=hi, extra=f'share of draws below 0.90: {pbelow:.2f}'))
    print(pd.DataFrame(ROB).round(4).to_string(index=False))
with section('C2. model-seed variance'):
    rows = []
    for sd in range(10):
        p_ = fit_lgb('reg', XL, yrL <= 2011, seed=sd); S_ = pv_sets(p_, true_scores(p_, yL)[caL]); r = dict(portfolio='Lending Club', seed=sd)
        for yy in (2013, 2014, 2015): m = yrL == yy; r[f'cov_{yy}'] = S_[m][np.arange(m.sum()), yL[m]].mean(); r[f'AUC_{yy}'] = roc_auc_score(yL[m], p_[m])
        rows.append(r)
        p2 = fit_predict('LGBM regularised', X0, yr <= 2008) if sd == 0 else None
    pr_rows = []
    for sd in range(10):
        LGB_REG['random_state'] = sd; p_ = fit_predict('LGBM regularised', X0, yr <= 2008); S_ = pv_sets(p_, true_scores(p_, y)[caP]); r = dict(portfolio='Prosper B', seed=sd)
        for yy in (2010, 2011): m = yr == yy; r[f'cov_{tlabel(yy)}'] = S_[m][np.arange(m.sum()), y[m]].mean(); r[f'AUC_{tlabel(yy)}'] = roc_auc_score(y[m], p_[m])
        pr_rows.append(r)
    LGB_REG['random_state'] = SEED; SEEDS = pd.concat([pd.DataFrame(rows), pd.DataFrame(pr_rows)]); save(SEEDS, 'C2_seed_variance')
    print(SEEDS.groupby('portfolio').agg(['mean', 'std']).drop(columns='seed').round(4).T.to_string())
with section('C3. dropping the earliest vintages'):
    rows = []
    for lab, mtr in (('LC train 2007-11 (paper)', yrL <= 2011), ('LC train 2009-11', (yrL >= 2009) & (yrL <= 2011))):
        p_ = fit_lgb('default', XL, mtr); S_ = pv_sets(p_, true_scores(p_, yL)[caL]); rows.append(dict(setting=lab, **{f'cov_{yy}': S_[yrL == yy][np.arange((yrL == yy).sum()), yL[yrL == yy]].mean() for yy in (2013, 2014, 2015)}, n_train=int(mtr.sum())))
    for lab, mtr in (('Prosper B train 2006-08 (paper)', yr <= 2008), ('Prosper B train 2007-08', (yr >= 2007) & (yr <= 2008))):
        p_ = fit_predict('LGBM default', X0, mtr); S_ = pv_sets(p_, true_scores(p_, y)[caP]); rows.append(dict(setting=lab, **{f'cov_{tlabel(yy)}': S_[yr == yy][np.arange((yr == yy).sum()), y[yr == yy]].mean() for yy in (2010, 2011)}, n_train=int(mtr.sum())))
    VIN = pd.DataFrame(rows); save(VIN, 'C3_vintage_sensitivity'); print(VIN.round(4).to_string(index=False))
with section('C4. profit outliers'):
    rows = []
    def gains(pi_, ap, te, rank_score, label, port):
        N = te.sum(); base = pi_[te].sum() / N; g1 = (pi_[te & ap].sum()) / N - base
        order = np.where(te)[0][np.argsort(-rank_score[te], kind='stable')]; k = int(0.9 * N); g2 = pi_[order[:k]].sum() / N - base
        rows.append(dict(portfolio=port, profit_version=label, static_refer_to_reject_gain=g1, worst10pct_out_gain=g2, approve_all_profit=base))
    if HAS_LCPAY:
        S_ = pv_sets(pL, scL[caL]); ap = S_[:, 0] & ~S_[:, 1]; te = yrL >= 2013; pr = LC['profit'].values; lo_, hi_ = np.percentile(pr, [1, 99])
        for lab, v in (('raw', pr), ('winsorised 1/99', np.clip(pr, lo_, hi_))): gains(v, ap, te, -pL, lab, 'Lending Club')
        keep = (pr >= lo_) & (pr <= hi_); gains(np.where(keep, pr, 0.0), ap & keep, te & keep, -pL, 'trimmed 1/99 (dropped)', 'Lending Club')
    apP = SP[:, 0] & ~SP[:, 1]; teP = np.isin(yr, [2010, 2011]); lo_, hi_ = np.percentile(pi, [1, 99])
    for lab, v in (('raw', pi), ('winsorised 1/99', np.clip(pi, lo_, hi_))): gains(v, apP, teP, -pP, lab, 'Prosper B')
    keep = (pi >= lo_) & (pi <= hi_); gains(np.where(keep, pi, 0.0), apP & keep, teP & keep, -pP, 'trimmed 1/99 (dropped)', 'Prosper B')
    PRO = pd.DataFrame(rows); save(PRO, 'C4_profit_outliers'); print(PRO.round(2).to_string(index=False)); print("Conclusion holds if every gain is negative (rejecting does not beat approving everyone).")


========== C1. calibration + month-block resampling ==========


                                         check    portfolio period  estimate     lo     hi                           extra
coverage interval (calibration + month blocks) Lending Club   2013    0.9178 0.9148 0.9212 share of draws below 0.90: 0.00
coverage interval (calibration + month blocks) Lending Club   2014    0.9025 0.8988 0.9069 share of draws below 0.90: 0.09
coverage interval (calibration + month blocks) Lending Club   2015    0.8900 0.8864 0.8936 share of draws below 0.90: 1.00
coverage interval (calibration + month blocks)    Prosper B   2010    0.9137 0.8941 0.9251 share of draws below 0.90: 0.12
coverage interval (calibration + month blocks)    Prosper B 2011Q1    0.9258 0.8989 0.9519 share of draws below 0.90: 0.03
coverage interval (calibration + month blocks)    Prosper A   2009    0.9472 0.9393 0.9545 share of draws below 0.90: 0.00
coverage interval (calibration + month blocks)    Prosper A   2010    0.9553 0.9452 0.9660 share of draws below 0.90: 0.00
coverage interva

portfolio        Lending Club  Prosper B
cov_2013   mean        0.9166        NaN
           std         0.0004        NaN
AUC_2013   mean        0.6536        NaN
           std         0.0004        NaN
cov_2014   mean        0.8985        NaN
           std         0.0001        NaN
AUC_2014   mean        0.6516        NaN
           std         0.0006        NaN
cov_2015   mean        0.8829        NaN
           std         0.0003        NaN
AUC_2015   mean        0.6616        NaN
           std         0.0006        NaN
cov_2010   mean           NaN     0.9084
           std            NaN     0.0019
AUC_2010   mean           NaN     0.6759
           std            NaN     0.0014
cov_2011Q1 mean           NaN     0.9201
           std            NaN     0.0044
AUC_2011Q1 mean           NaN     0.6912
           std            NaN     0.0038
[C2. model-seed variance: 32s | total 0.9 min]

========== C3. dropping the earliest vintages ==========


                        setting  cov_2013  cov_2014  cov_2015  n_train  cov_2010  cov_2011Q1
       LC train 2007-11 (paper)    0.9178    0.9025    0.8900    28243       NaN         NaN
               LC train 2009-11    0.9166    0.8993    0.8848    26438       NaN         NaN
Prosper B train 2006-08 (paper)       NaN       NaN       NaN    28913    0.9137      0.9258
        Prosper B train 2007-08       NaN       NaN       NaN    23011    0.9041      0.9164
[C3. dropping the earliest vintages: 5s | total 1.0 min]

========== C4. profit outliers ==========


   portfolio         profit_version  static_refer_to_reject_gain  worst10pct_out_gain  approve_all_profit
Lending Club                    raw                      -119.46               -75.05             1055.85
Lending Club        winsorised 1/99                      -123.66               -77.97             1090.36
Lending Club trimmed 1/99 (dropped)                      -127.40               -80.58             1164.60
   Prosper B                    raw                      -242.60               -55.22              610.05
   Prosper B        winsorised 1/99                      -238.86               -53.69              607.44
   Prosper B trimmed 1/99 (dropped)                      -229.24               -49.14              596.73
Conclusion holds if every gain is negative (rejecting does not beat approving everyone).
[C4. profit outliers: 0s | total 1.0 min]


In [9]:
ROB = pd.DataFrame(ROB); save(ROB, 'C1_coverage_intervals'); AUDT = pd.DataFrame(AUD); save(AUDT, 'A_audit_items')
lines = [f"AUDIT SUMMARY ({time.strftime('%Y-%m-%d %H:%M')}), failed sections: {FAILED or 'none'}", "", "## Audit items", AUDT.to_string(index=False), "", "## Manuscript number checks", CHK.to_string(index=False) if len(CHK) else '(none)', "", "## Coverage intervals", ROB.round(4).to_string(index=False)]
(OUT/'audit_summary.txt').write_text("\n".join(lines)); print("\n".join(lines)[:3000]); print(f"\nFAILED SECTIONS: {FAILED or 'none'}")
print("Send me this file:", shutil.make_archive(str(Q2/'outputs_audit'), 'zip', OUT))

AUDIT SUMMARY (2026-11-08 08:25), failed sections: none

## Audit items
part                                                      item                     value                                                                               note
A-LC                loans (36-month, resolved, issued <= 2015)                    582113                                                                manuscript: 582,113
A-LC                                              default rate                  0.134842                                                                 manuscript: 13.48%
A-LC                                        duplicate loan ids                         0                                                                                   
A-LC                          rows with a missing core feature                         0                                                                                   
A-LC                             fully duplicated feature rows      